📝 Python Quiz &nbsp;·&nbsp; *Bài 4/4*

[« 📝 Python Modules Quiz — NumPy · Pandas · SciPy · Matplotlib · ML · DSA · MySQL · MongoDB · Django](03_modules_quiz.ipynb) &nbsp;•&nbsp; [📚 Mục lục](../README.md) &nbsp;•&nbsp; Hoàn thành 🎉

# 🔁 Ôn tập ngẫu nhiên (Spaced Repetition)

Notebook này rút **câu hỏi ngẫu nhiên** từ **ngân hàng câu hỏi** của toàn bộ khóa học (file [`question_bank.json`](question_bank.json) — gồm mọi câu 📝 Exercise cuối các bài học) để bạn ôn lại.

## 🧠 Vì sao ôn tập ngắt quãng giúp nhớ lâu?

Theo **đường cong quên lãng (Ebbinghaus)**, ta quên rất nhanh những gì vừa học nếu không ôn lại. Ôn lại **đúng lúc sắp quên** giúp kiến thức đi vào trí nhớ dài hạn.

Notebook dùng **hệ thống Leitner** — chia câu hỏi vào 5 "hộp":

```text
Hộp 1  ──đúng──▶ Hộp 2 ──đúng──▶ Hộp 3 ──đúng──▶ Hộp 4 ──đúng──▶ Hộp 5 (đã thuộc)
  ▲                                                                  │
  └──────────────────────────── sai ─────────────────────────────────┘
```

- Câu **trả lời sai** → quay về **hộp 1** → được hỏi lại **thường xuyên**.
- Câu **trả lời đúng** → lên hộp cao hơn → được hỏi **thưa dần**.
- Tiến độ lưu trong file `review_history.json` (cùng thư mục, không đưa lên Git).

> 💡 **Lịch ôn gợi ý:** học bài mới → ôn sau **1 ngày** → **3 ngày** → **7 ngày** → **14 ngày**. Mỗi lần chỉ cần 10 câu, khoảng 5 phút.

## ⚙️ Bước 1 — Cấu hình

In [ ]:
# Chương muốn ôn — để trống [] nếu muốn ôn tất cả
CHAPTERS = ["01_python_tutorial", "02_python_classes", "03_file_handling"]

# Chỉ ôn tới bài số mấy trong chuỗi bài 01 → 03 (1–124). Ví dụ đã học tới bài 045: UP_TO = 45
UP_TO = 124

# Số câu mỗi lượt
N = 10

# True: nhập đáp án từng câu · False: chế độ flashcard (xem câu hỏi, tự nhẩm, mở đáp án)
INTERACTIVE = True

Các chương có trong ngân hàng câu hỏi:

In [ ]:
import json
from collections import Counter
from pathlib import Path

bank = json.loads(Path("question_bank.json").read_text(encoding="utf-8"))
for chapter, count in Counter(q["chapter"] for q in bank).items():
    print(f"{chapter:<28}{count:>4} câu")
print(f"{'Tổng':<28}{len(bank):>4} câu")

## 🎲 Bước 2 — Chọn câu hỏi

In [ ]:
import random

HISTORY = Path("review_history.json")
CORE = ("01_python_tutorial", "02_python_classes", "03_file_handling")

try:
    history = json.loads(HISTORY.read_text(encoding="utf-8"))
except (FileNotFoundError, json.JSONDecodeError):
    history = {}

pool = [
    q for q in bank
    if (not CHAPTERS or q["chapter"] in CHAPTERS)
    and (q["chapter"] not in CORE or q["order"] <= UP_TO)
]

def box_of(q):
    return history.get(str(q["id"]), 1)

# Hộp càng thấp → trọng số càng cao (hộp 1: 16, hộp 5: 1)
def priority(q):
    return random.random() ** (1 / 2 ** (5 - box_of(q)))

selected = sorted(pool, key=priority, reverse=True)[:N]
print(f"Chọn {len(selected)} / {len(pool)} câu phù hợp.")
print("Phân bố hộp Leitner:", dict(sorted(Counter(box_of(q) for q in pool).items())))

## ✍️ Bước 3 — Làm bài

Chạy cell dưới. Với câu trắc nghiệm, nhập **A/B/C/D**; với câu điền khuyết, nhập phần còn thiếu (nhiều chỗ trống thì cách nhau bằng dấu cách).

In [ ]:
try:
    from IPython.display import Markdown, display

    def show(md):
        display(Markdown(md))
except ImportError:
    show = print

LETTERS = "ABCDEFGH"

def normalize(text):
    return " ".join(text.replace(",", " ").split()).lower()

def ask(q, number):
    text = f"**Câu {number}/{len(selected)}** — *{q['lesson']}*\n\n{q['question']}"
    if q["options"]:
        text += "\n\n" + "\n".join(f"- **{LETTERS[i]}.** {o}" for i, o in enumerate(q["options"]))
        right = q["correct"]
    else:
        right = " ".join(q["answers"])
    show(text)

    if INTERACTIVE:
        answer = input("👉 Đáp án của bạn: ")
        ok = normalize(answer) == normalize(right)
        show(("✅ **Chính xác!**" if ok else f"❌ **Chưa đúng** — đáp án: `{right}`")
             + (f"\n\n💡 {q['explanation']}" if q["explanation"] else ""))
    else:
        ok = None
        show(f"<details><summary>👉 Xem đáp án</summary>\n\n`{right}`\n\n{q['explanation']}\n\n</details>")
    return ok

results = []
for i, q in enumerate(selected, 1):
    ok = ask(q, i)
    results.append((q, ok))
    if ok is not None:
        history[str(q["id"])] = min(box_of(q) + 1, 5) if ok else 1
    show("---")

HISTORY.write_text(json.dumps(history, indent=1), encoding="utf-8")

## 📊 Bước 4 — Kết quả & bài cần ôn lại

In [ ]:
answered = [(q, ok) for q, ok in results if ok is not None]
if answered:
    score = sum(ok for _, ok in answered)
    print(f"🎯 Điểm: {score}/{len(answered)}")
    wrong = {}
    for q, ok in answered:
        if not ok:
            wrong[q["lesson"]] = q["link"]
    if wrong:
        show("### 📚 Nên ôn lại các bài:\n\n" + "\n".join(f"- [{t}]({link})" for t, link in wrong.items()))
    else:
        print("🏆 Tuyệt vời! Hãy quay lại sau vài ngày để ôn tiếp.")

mastered = sum(1 for q in pool if box_of(q) == 5)
print(f"📦 Đã thuộc (hộp 5): {mastered}/{len(pool)} câu")

> 🔄 Muốn làm lại từ đầu? Xoá file `review_history.json`.

## 🃏 Bonus: Học bằng Anki

Toàn bộ ngân hàng câu hỏi được xuất sẵn thành bộ thẻ **Anki** trong file [`flashcards_anki.txt`](flashcards_anki.txt):

1. Cài [Anki](https://apps.ankiweb.net/) (miễn phí, có app điện thoại AnkiDroid/AnkiMobile).
2. **File → Import** → chọn `flashcards_anki.txt`.
3. Anki tự lên lịch ôn tập ngắt quãng cho bạn mỗi ngày. Thẻ được gắn tag theo chương (`python::01_python_tutorial::...`).

---

[« 📝 Python Modules Quiz — NumPy · Pandas · SciPy · Matplotlib · ML · DSA · MySQL · MongoDB · Django](03_modules_quiz.ipynb) &nbsp;•&nbsp; [📚 Mục lục](../README.md) &nbsp;•&nbsp; Hoàn thành 🎉